# Exercise Sheet 2 (programming part, solved)
*Machine Learning for Data Science, Winter Semester 2026/27 — Prof. Dr.-Ing. Gerhard Wunder, Freie Universität Berlin*

We first load a small handwritten digits dataset provided as part of the scikit-learn library and a few relevant libraries.

In [1]:
import matplotlib
%matplotlib inline
from matplotlib import pyplot as plt
import numpy
import torch,torch.optim
import scipy,scipy.spatial
import sklearn,sklearn.datasets

dataset = sklearn.datasets.load_digits()
X = dataset['data']
T = dataset['target']
R = numpy.random.mtrand.RandomState(0).permutation(len(X))[:750] # randomly select 750
X = X[R]
T = T[R]

## Exercise 3: Implementing Metric MDS (15+15+10 P)

In this exercise, we will implement a low-dimensional embedding technique, multi-dimensional scaling (MDS), apply it to the simple handwritten digits dataset above, and visualize the embedded data using a series of scatter plots.

The MDS algorithm does not operate with the dataset directly, but instead, with a matrix of distances. Also, in order to be rather independent from the scale and dimensionality of the data, we would like to normalize the input distance such that distances are on average 10. Lastly, the distance should be provided as a torch tensor so that it can be used for the gradient-based training procedure.

**(a)** Write a function that takes the data as input and returns a distance matrix according to the specifications above.

In [2]:
X[0]

array([ 0.,  0., 11., 16., 15.,  3.,  0.,  0.,  0.,  5., 16., 12., 11.,
       13.,  0.,  0.,  0.,  3., 13.,  1.,  5., 15.,  0.,  0.,  0.,  0.,
        0.,  0., 12., 11.,  0.,  0.,  0.,  0.,  0.,  1., 16.,  7.,  0.,
        0.,  0.,  0.,  0., 10., 15.,  0.,  0.,  0.,  0.,  0., 12., 16.,
       16., 11.,  1.,  0.,  0.,  0., 13., 13.,  8., 13., 16.,  8.])

In [3]:
X[1].shape

(64,)

In [4]:
y = numpy.arange(15).reshape(3, 5)
def my_pairwise_distance(y):
    y1 = y[numpy.newaxis, :]
    y2 = y[:, numpy.newaxis]
    diff = y1 - y2
    sums = numpy.sum(diff ** 2, axis=2)
    pairwise_np = numpy.sqrt(sums)
    pairwise_sklearn = sklearn.metrics.pairwise_distances(y)
    assert numpy.array_equal(pairwise_np, pairwise_sklearn)
    return pairwise_np

In [5]:
dist = my_pairwise_distance(y)
dist

array([[ 0.        , 11.18033989, 22.36067977],
       [11.18033989,  0.        , 11.18033989],
       [22.36067977, 11.18033989,  0.        ]])

In [6]:
print((dist - dist.mean() + 10))
print((dist - dist.mean() + 10).mean())

[[ 0.0619201  11.24225999 22.42259987]
 [11.24225999  0.0619201  11.24225999]
 [22.42259987 11.24225999  0.0619201 ]]
10.0


In [7]:
my_pairwise_distance(X).mean()

np.float64(48.22396884373512)

In [8]:
def normalize_min0_mean10(arr, target=10.0):
    shifted = arr - arr.min()
    return shifted * (target / shifted.mean())

In [9]:
def getDistanceMatrix(X, target_mean = 10):

    distances = my_pairwise_distance(X)
    norm_distances = normalize_min0_mean10(distances)
    return torch.tensor(norm_distances)

D = getDistanceMatrix(X)

print(f"shape: {str(D.shape)}")
print(f"min:   {D.min():6.3f}")
print(f"mean:  {D.mean():6.3f}")
print(f"max:   {D.max():6.3f}")

shape: torch.Size([750, 750])
min:    0.000
mean:  10.000
max:   15.391


Expected output:
```
shape: torch.Size([750, 750])
min:    0.000
mean:  10.000
max:   15.391
```

We now would like learn Metric MDS. We will consider a simple gradient-based implementation. The outline of the gradient-based procedure is provided in the code below. The only part missing is the computation of the objective $J(Y)$ where $Y$ are the matrix of size $N \times 2$ containing the coordinates of data points in embedded space. As a reminder, the metric MDS objective is given by:
$$ J(\boldsymbol{y}_1,\dots,\boldsymbol{y}_N) = \left(\frac{\sum_{i<j} \big(d_{ij} - \|\boldsymbol{y}_i - \boldsymbol{y}_j\|\big)^2}{\sum_{i<j} d_{ij}^2}\right)^{1/2} $$

**(b)** Complete the code below by expressing `J` as a function of `Y` and `D`.

*(Hint: Running the training procedure may take up to 5 minutes. For debugging purposes, you can temporarily reduce the number of iterations.)*

In [ ]:
def getObjective(Y,D):

    # ------------------------------------
    # TODO: Replace by your code
    # ------------------------------------
    # Return the metric MDS objective J(Y) given embedding Y and distances D
    raise NotImplementedError
    # ------------------------------------

Y = torch.randn([len(X),2])*5
Y.requires_grad_(True)

optimizer = torch.optim.SGD((Y,), lr=100.0, momentum=0.95)

Yhist = []

for i in range(10**4+1):
    optimizer.zero_grad()

    J = getObjective(Y,D)

    J.backward()

    # Keep track of the objective
    if i in numpy.arange(11)**4:
        print(f"{i:5d} {J.data.numpy():.3f}")
        Yhist += [Y.data.numpy()*1.0]

    optimizer.step()

Expected output (up to the random initialization):
```
    0 0.494
    1 0.494
   16 0.461
   81 0.423
  256 0.409
  625 0.360
 1296 0.338
 2401 0.337
 4096 0.336
 6561 0.335
10000 0.334
```

The code above has stored the embedding at 11 distinct checkpoints (available in `Yhist`). This allows to visualize how the embedding (initialized at random) evolves throughout training, and how it progressively improves the representation of distances in the data.

**(c)** Using matplotlib, plot the embedding at checkpoints {0,3,4,5,6,10}, and color-code the embedding according to the digit's class. *(Hint: because there are 10 classes, you can use the colormap `tab10` in matplotlib.)*

In [ ]:
def plotEmbeddings(Yhist,T):

    # ------------------------------------
    # TODO: Replace by your code
    # ------------------------------------
    # Scatter plots of the embedding at checkpoints {0,3,4,5,6,10}, colored by class T (cmap 'tab10')
    raise NotImplementedError
    # ------------------------------------

plotEmbeddings(Yhist,T)

We observe that during training, data points travel in embedded space. In the final configuration, embedded data forms clusters of points of same class.